# Stabilized SU(2) reblocking for the S4 indicator proof

This notebook checks the proposed repair of the proof in `essay/S4_indicator.tex`.

Logical point: for a unitary `S4` sewing matrix, a gauge transformation acts as
`B'(K)=G(K)^dagger B(K)G(K)` at an `S4`-invariant momentum `K`, so `det B(K)` is gauge invariant. Therefore the strict `U(2) -> SU(2)` reduction is allowed only for blocks whose determinant obstruction is absent at the fixed points. In the reduced double-valued row order used below, this means that the multiplicities of the two conjugate `S4` pairs match at every `S4`-fixed momentum.

The stabilized check is the integer problem

`safe_blocks * U = target + EBR * V`, with `U,V >= 0` integral.

Equivalently, `target = allowed_blocks * U - EBR * V`: after adding atomic bands to the target, it can be reblocked into scalar one-band blocks and determinant-one two-band blocks. The scalar one-band blocks are **not** literal `SU(1)` blocks in this spinful convention; they are retained only because a 1x1 sewing matrix contributes identically zero to the cubic Chern-Simons 3-form.

In [1]:
from __future__ import annotations

import re
import sys
import time
from itertools import product
from pathlib import Path

import numpy as np


def find_data_dir() -> Path:
    """Find essay_script whether the notebook is launched there or from the repo root."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "essay_script", cwd.parent / "essay_script"]
    for candidate in candidates:
        if (candidate / "ebr_raw_data8133.txt").exists() and (candidate / "gu_cover_solver.py").exists():
            return candidate
    raise FileNotFoundError("Could not find essay_script data files from current working directory")


DATA_DIR = find_data_dir()
sys.path.insert(0, str(DATA_DIR))

N_COLS = 36
FULL_N_ROWS = 40
N_ROWS = 20
ROW_OFFSET = {"Gamma": 0, "A": 8, "M": 16, "Z": 24, "R": 32, "X": 36}
KPOINT_TYPES = ("Gamma", "A", "M", "Z", "R", "X")
OLD_ROWS = (
    list(range(4, 8))
    + list(range(12, 16))
    + list(range(20, 24))
    + list(range(28, 32))
    + [34, 35]
    + [38, 39]
)

BLOCKS = [
    ("Gamma", [0, 1, 2, 3]),
    ("A", [4, 5, 6, 7]),
    ("M", [8, 9, 10, 11]),
    ("Z", [12, 13, 14, 15]),
    ("R", [16, 17]),
    ("X", [18, 19]),
]
S4_BLOCKS = BLOCKS[:4]
C2_BLOCKS = BLOCKS[4:]

row_labels = (
    [f"Gamma_{i}" for i in range(5, 9)]
    + [f"A_{i}" for i in range(5, 9)]
    + [f"M_{i}" for i in range(5, 9)]
    + [f"Z_{i}" for i in range(5, 9)]
    + [f"R_{i}" for i in range(3, 5)]
    + [f"X_{i}" for i in range(3, 5)]
)


def parse_cell(cell_html: str) -> list[int]:
    """Extract representation sub-indices from one BCS table cell."""
    parts = re.split(r"&nbsp;&oplus;&nbsp;|&oplus;", cell_html)
    indices = []
    for part in parts:
        match = re.search(r"<sub>(\d+)</sub>", part)
        if match:
            indices.append(int(match.group(1)))
    return indices


def _row_label(row_html: str) -> str:
    cells = re.findall(r"<td[^>]*>(.*?)</td>", row_html, re.DOTALL)
    if not cells:
        return ""
    return re.sub(r"<[^>]+>", "", cells[0]).strip()


def _identify_kpoint(label: str) -> str | None:
    label = label.replace("?", "Gamma").replace("&Gamma;", "Gamma")
    for rep in KPOINT_TYPES:
        if re.match(rf"^{rep}\s*:", label):
            return rep
    return None


def _extract_table_rows(html: str) -> list[str]:
    # The copied BCS HTML may omit closing </tr> tags, so split on opening tags.
    return [part for part in re.split(r"<tr>", html, flags=re.IGNORECASE) if "<td" in part]


def build_B_from_html(html: str) -> np.ndarray:
    data_rows: list[tuple[str, str]] = []
    for row_html in _extract_table_rows(html):
        rep_type = _identify_kpoint(_row_label(row_html))
        if rep_type is not None:
            data_rows.append((rep_type, row_html))

    if len(data_rows) != 6:
        labels = [_row_label(row) for _, row in data_rows]
        raise ValueError(f"Expected 6 k-point rows, found {len(data_rows)}: {labels}")

    first_cells = re.findall(r"<td[^>]*>(.*?)</td>", data_rows[0][1], re.DOTALL)
    n_label_cols = len(first_cells) - N_COLS
    if n_label_cols < 1:
        raise ValueError(f"Expected {N_COLS} data columns, got {len(first_cells)} cells")

    B = np.zeros((FULL_N_ROWS, N_COLS), dtype=int)
    for col_idx in range(N_COLS):
        for rep_type, row_html in data_rows:
            cells = re.findall(r"<td[^>]*>(.*?)</td>", row_html, re.DOTALL)
            cell_html = cells[n_label_cols + col_idx]
            for sub_idx in parse_cell(cell_html):
                B[ROW_OFFSET[rep_type] + sub_idx - 1, col_idx] = 1
    return B


raw_path = DATA_DIR / "ebr_raw_data8133.txt"
EBR_full = build_B_from_html(raw_path.read_text(encoding="utf-8"))
EBR = EBR_full[OLD_ROWS, :].copy()

# Compatibility matrix in the same 20-row basis as Appendix B.
CR = np.zeros((8, N_ROWS), dtype=int)
CR[0, 0] = CR[0, 1] = 1
CR[0, 12] = CR[0, 13] = -1
CR[1, 2] = CR[1, 3] = 1
CR[1, 14] = CR[1, 15] = -1
CR[2, 4] = CR[2, 5] = 1
CR[2, 8] = CR[2, 9] = -1
CR[3, 6] = CR[3, 7] = 1
CR[3, 10] = CR[3, 11] = -1
CR[4, 16] = 1
CR[4, 18] = -1
CR[5, 17] = 1
CR[5, 19] = -1
CR[6, 0:4] = 1
CR[6, 16] = CR[6, 17] = -1
CR[7, 4:8] = 1
CR[7, 16] = CR[7, 17] = -1

# Indicator rows from band_decomposition.ipynb / find_basis.py.
# P_1 is scaled by 2, so z_4S = 0 is P_1 dot B = 0 mod 8.
P_1 = np.array([0, 0, 0, 0, 1.5, -0.5, -1.5, 0.5, 0, 0, 0, 0, 1.5, -0.5, -1.5, 0.5, -1.0, 1.0, 0, 0])
P_2 = np.array([-1, 0, 1, 0, 1, 0, -1, 0, -1, 0, 1, 0, 1, 0, -1, 0, 0, 0, 0, 0], dtype=int)
P_1 = (2 * P_1).astype(int)

print("data directory:", DATA_DIR)
print("EBR shape:", EBR.shape)
print("CR @ EBR == 0:", bool(np.all(CR @ EBR == 0)))
print("row labels:", row_labels)

data directory: D:\yao\PkU\Song Group\Symmetry Representation and Topological Invariant\essay_script
EBR shape: (20, 36)
CR @ EBR == 0: True
row labels: ['Gamma_5', 'Gamma_6', 'Gamma_7', 'Gamma_8', 'A_5', 'A_6', 'A_7', 'A_8', 'M_5', 'M_6', 'M_7', 'M_8', 'Z_5', 'Z_6', 'Z_7', 'Z_8', 'R_3', 'R_4', 'X_3', 'X_4']


## Enumerate one-band and two-band blocks

The targets are exactly the 1-band and 2-band symmetry-data vectors satisfying compatibility plus `z_4S=delta_2S=0`, following Appendix B.

For the two-band `SU(2)` criterion, the determinant obstruction must vanish at both the `S4`- and `C2`-invariant momenta. In each `S4`-fixed block `(K_5,K_6,K_7,K_8)` at `Gamma,Z,M,A`, the conjugate pairs are `(K_5,K_7)` and `(K_6,K_8)`, so these pair multiplicities must match. At `R` and `X`, the conjugate `C2` pair multiplicities `(K_3,K_4)` must also match, so `det D(R)=det D(X)=1`. No one-band block is classified as `SU(1)` below, since its fixed-point eigenvalues are not all equal to `1`.

In [2]:
def compositions(total: int, parts: int):
    """Non-negative integer tuples of length parts summing to total."""
    if parts == 1:
        yield (total,)
        return
    for first in range(total + 1):
        for rest in compositions(total - first, parts - 1):
            yield (first,) + rest


def enumerate_band_data(N: int, mods=(8, 2)) -> np.ndarray:
    """All N-band B >= 0 satisfying compatibility, z_4S=0, and delta_2S=0."""
    block_choices = [list(compositions(N, len(idx))) for _, idx in BLOCKS]
    results = []
    for combo in product(*block_choices):
        B = np.zeros(N_ROWS, dtype=int)
        for (_, idx), vals in zip(BLOCKS, combo):
            B[idx] = vals
        if np.any(CR @ B != 0):
            continue
        if (P_1 @ B) % mods[0] != 0:
            continue
        if (P_2 @ B) % mods[1] != 0:
            continue
        results.append(B)
    return np.array(results, dtype=int).reshape(-1, N_ROWS)


def z2_numerator(B: np.ndarray) -> int:
    B = np.asarray(B, dtype=int).reshape(-1)
    return int(B[1] + B[5] + B[9] + B[13] - (B[0] + B[4] + B[8] + B[12]))


def z2(B: np.ndarray) -> int:
    num = z2_numerator(B)
    if num % 2 != 0:
        raise ValueError(f"z2 numerator is odd for {B}: {num}")
    return (num // 2) % 2


def is_su2_candidate(B: np.ndarray) -> bool:
    """det B(K)=1 at Gamma, Z, M, A and det D(K)=1 at R, X."""
    B = np.asarray(B, dtype=int).reshape(-1)
    for _, idx in S4_BLOCKS:
        if B[idx[0]] != B[idx[2]]:  # K_5 paired with K_7
            return False
        if B[idx[1]] != B[idx[3]]:  # K_6 paired with K_8
            return False
    for _, idx in C2_BLOCKS:
        if B[idx[0]] != B[idx[1]]:  # K_3 paired with K_4
            return False
    return True


def has_equal_c2_pair_somewhere(B: np.ndarray) -> bool:
    """Diagnostic used in the TeX: both bands have the same C2 eigenvalue at some S4-fixed K."""
    B = np.asarray(B, dtype=int).reshape(-1)
    for _, idx in S4_BLOCKS:
        if B[idx[0]] + B[idx[1]] == 2:
            return True
        if B[idx[2]] + B[idx[3]] == 2:
            return True
    return False


def active_rep_labels(B: np.ndarray) -> list[str]:
    B = np.asarray(B, dtype=int).reshape(-1)
    return [f"{row_labels[i]}={B[i]}" for i in np.nonzero(B)[0]]


B1 = enumerate_band_data(1)
B2 = enumerate_band_data(2)
su2_mask = np.array([is_su2_candidate(b) for b in B2], dtype=bool)
B2_su2 = B2[su2_mask]
su2_block_ids = np.where(su2_mask)[0]

print("B1 count:", len(B1))
print("B2 count:", len(B2))
print("B1 z2 values:", sorted({z2(b) for b in B1}))
print("B2 z2 counts:", {value: int(sum(z2(b) == value for b in B2)) for value in (0, 1)})
print("SU(2)-candidate B2 count:", len(B2_su2))
print("SU(2)-candidate B2 z2 counts:", {value: int(sum(is_su2_candidate(b) and z2(b) == value for b in B2)) for value in (0, 1)})
print("z2=1 B2 blocks not directly SU(2)-reducible:", int(sum((not is_su2_candidate(b)) and z2(b) == 1 for b in B2)))
print("z2=1 B2 blocks with equal C2 pair at some fixed K:", int(sum(has_equal_c2_pair_somewhere(b) and z2(b) == 1 for b in B2)))

B1 count: 16
B2 count: 438
B1 z2 values: [0]
B2 z2 counts: {0: 294, 1: 144}
SU(2)-candidate B2 count: 16
SU(2)-candidate B2 z2 counts: {0: 8, 1: 8}
z2=1 B2 blocks not directly SU(2)-reducible: 136
z2=1 B2 blocks with equal C2 pair at some fixed K: 48


In [3]:
# Are EBRs SU(1) or SU(2)?
[is_su2_candidate(b) for b in EBR.T]
# Well, some are but some are not.

[True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 True,
 False,
 False]

## Stabilized integer decomposition

The generator set `allowed_blocks` contains all 16 scalar one-band blocks plus the determinant-one two-band blocks. Here "allowed" means "allowed in the invariant calculation": scalar 1x1 blocks are kept as zero-contribution `U(1)` blocks, while only the determinant-one 2x2 blocks enter the `SU(2)` degree argument. A second, stronger check uses only determinant-one two-band generators for the 2D targets.

In [4]:
from gu_cover_solver import build_highs_solver, solve_one_warm


def solve_targets(targets: np.ndarray, generators: np.ndarray, ebr: np.ndarray):
    """Solve generators @ U = target + ebr @ V for every target."""
    targets = np.asarray(targets, dtype=int)
    generators = np.asarray(generators, dtype=int)
    ebr = np.asarray(ebr, dtype=int)

    nU = generators.shape[1]
    A = np.ascontiguousarray(np.hstack([generators, -ebr]), dtype=np.float64)
    solver, row_idx = build_highs_solver(A, N_ROWS, A.shape[1])

    feasible = np.zeros(len(targets), dtype=bool)
    U_all = np.zeros((len(targets), nU), dtype=int)
    V_all = np.zeros((len(targets), ebr.shape[1]), dtype=int)
    failures: list[int] = []

    for i, target in enumerate(targets):
        ok, U, V = solve_one_warm(solver, row_idx, N_ROWS, nU, target)
        if ok:
            assert np.all(generators @ U - ebr @ V == target)
            feasible[i] = True
            U_all[i] = U
            V_all[i] = V
        else:
            failures.append(i)
    return feasible, U_all, V_all, failures


t0 = time.time()
# Columns are generators. B1 blocks are scalar U(1) zero-contribution blocks, not SU(1) blocks.
allowed_blocks = np.vstack([B1, B2_su2]).T
all_targets = np.vstack([B1, B2])
feasible_allowed, U_allowed, V_allowed, failures_allowed = solve_targets(all_targets, allowed_blocks, EBR)

su2_blocks_only = B2_su2.T
feasible_b2_su2, U_b2_su2, V_b2_su2, failures_b2_su2 = solve_targets(B2, su2_blocks_only, EBR)
feasible_b1_su2, U_b1_su2, V_b1_su2, failures_b1_su2 = solve_targets(B1, su2_blocks_only, EBR)

print("allowed block matrix shape:", allowed_blocks.shape)
print("all 1D+2D targets feasible with scalar U(1)+SU(2) blocks:", bool(feasible_allowed.all()), f"({feasible_allowed.sum()}/{len(all_targets)})")
print("failures for scalar U(1)+SU(2) blocks:", failures_allowed)
print("all 2D targets feasible with SU(2) blocks only:", bool(feasible_b2_su2.all()), f"({feasible_b2_su2.sum()}/{len(B2)})")
print("failures for SU(2)-only 2D check:", failures_b2_su2)
print("all 1D targets feasible with SU(2) blocks only:", bool(feasible_b1_su2.all()), f"({feasible_b1_su2.sum()}/{len(B1)})")
print("failures for SU(2)-only 1D check:", failures_b1_su2)
print("max EBR columns added in scalar U(1)+SU(2) check:", int(V_allowed.sum(axis=1).max()))
print("max EBR columns added in SU(2)-only B2 check:", int(V_b2_su2.sum(axis=1).max()))
print("elapsed seconds:", round(time.time() - t0, 2))

allowed block matrix shape: (20, 32)
all 1D+2D targets feasible with scalar U(1)+SU(2) blocks: True (454/454)
failures for scalar U(1)+SU(2) blocks: []
all 2D targets feasible with SU(2) blocks only: True (438/438)
failures for SU(2)-only 2D check: []
all 1D targets feasible with SU(2) blocks only: False (0/16)
failures for SU(2)-only 1D check: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
max EBR columns added in scalar U(1)+SU(2) check: 2
max EBR columns added in SU(2)-only B2 check: 5
elapsed seconds: 18.18


## Counterexample report and one explicit reblocking

If the failure lists above are empty, there are no counterexamples among the enumerated 1D and 2D blocks. The cell below prints one representative two-band target that is not directly `SU(2)`-reducible, and shows how the integer solve reblocks it after adding EBRs.

In [5]:
if failures_allowed or failures_b2_su2:
    print("Counterexamples found.")
    if failures_allowed:
        print("scalar U(1)+SU(2) failed target indices:", failures_allowed)
    if failures_b2_su2:
        print("SU(2)-only B2 failed target indices:", failures_b2_su2)
else:
    print("No counterexamples found among all 16 one-band and 438 two-band blocks.")

# Show an explicit non-SU(2) z2=1 two-band block and its stabilized SU(2)-only decomposition.
example_candidates = [i for i, b in enumerate(B2) if (not is_su2_candidate(b)) and z2(b) == 1]
example_idx = example_candidates[0]
example = B2[example_idx]
U = U_b2_su2[example_idx]
V = V_b2_su2[example_idx]
used_su2_local = np.nonzero(U)[0]
used_ebr = np.nonzero(V)[0]

print("\nRepresentative target: B2 index", example_idx, "(0-based), z2=", z2(example), "direct SU(2)=", is_su2_candidate(example))
print("target nonzero entries:", active_rep_labels(example))
print("added EBR columns (1-based, multiplicity):", [(int(j + 1), int(V[j])) for j in used_ebr])
print("SU(2) generator B2 indices used (0-based B2 index, multiplicity):", [(int(su2_block_ids[j]), int(U[j])) for j in used_su2_local])

lhs = su2_blocks_only @ U
rhs = example + EBR @ V
print("decomposition verified:", bool(np.array_equal(lhs, rhs)))
print("both sides nonzero entries:", active_rep_labels(lhs))

No counterexamples found among all 16 one-band and 438 two-band blocks.

Representative target: B2 index 26 (0-based), z2= 1 direct SU(2)= False
target nonzero entries: ['Gamma_8=2', 'A_6=2', 'M_5=1', 'M_6=1', 'Z_7=1', 'Z_8=1', 'R_4=2', 'X_4=2']
added EBR columns (1-based, multiplicity): [(15, 1), (22, 1), (36, 1)]
SU(2) generator B2 indices used (0-based B2 index, multiplicity): [(133, 1), (160, 1), (310, 1)]
decomposition verified: True
both sides nonzero entries: ['Gamma_5=1', 'Gamma_6=2', 'Gamma_7=1', 'Gamma_8=2', 'A_5=1', 'A_6=2', 'A_7=1', 'A_8=2', 'M_5=1', 'M_6=2', 'M_7=1', 'M_8=2', 'Z_5=2', 'Z_6=1', 'Z_7=2', 'Z_8=1', 'R_3=3', 'R_4=3', 'X_3=3', 'X_4=3']


In [6]:
print(EBR[:,14])
print(EBR[:,21])
print(EBR[:,35])
print(B2[133])
print(B2[160])
print(B2[310])

[0 1 0 0 1 0 0 0 0 1 0 0 1 0 0 0 1 0 1 0]
[0 0 1 0 0 0 0 1 0 0 0 1 0 0 1 0 1 0 1 0]
[1 1 0 0 0 0 1 1 0 0 1 1 1 1 0 0 1 1 1 1]
[0 1 0 1 0 1 0 1 1 0 1 0 0 1 0 1 1 1 1 1]
[0 1 0 1 1 0 1 0 0 1 0 1 1 0 1 0 1 1 1 1]
[1 0 1 0 0 1 0 1 0 1 0 1 1 0 1 0 1 1 1 1]


## Conclusion

Within the finite Appendix-B enumeration, the repair is feasible if one treats 1x1 blocks as scalar zero-contribution `U(1)` blocks, not as literal `SU(1)` blocks: every admissible one-band or two-band block can be stabilized by atomic EBRs and rewritten as a sum of scalar blocks and determinant-one two-band blocks. In fact, every two-band target can be stabilized and rewritten using determinant-one two-band blocks only, while the 16 one-band targets cannot be rewritten using only determinant-one two-band blocks plus EBRs. This supports the revised proof strategy: keep the `SU(2)` degree argument only for determinant-one two-band blocks, and handle scalar blocks separately by their vanishing cubic contribution.